# 🌤️ Weather Assistant with openJiuwen (Interact)

本 notebook 演示如何用 **openJiuwen** 搭建一个支持中断恢复的智能天气助手 Agent：

1. 调用付费 API 前，先询问用户是否确认（单节点交互中断）
2. 并发询问「城市」和「温度单位」（多节点交互中断）
3. 调用外部 API 时可能网络超时 / 限额，异常后重试只重新跑失败节点（异常中断恢复）

全程保持 **session_id** 不变，即可在任何时刻断点续跑。

In [1]:
# 首次运行需安装依赖
#%pip install jiuwen aiohttp

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement jiuwen (from versions: none)
ERROR: No matching distribution found for jiuwen


In [2]:
import uuid, random

from openjiuwen.core.context_engine import ModelContext
from openjiuwen.core.session import InteractiveInput
from openjiuwen.core.session.node import Session
from openjiuwen.core.workflow import (
    ComponentComposable,
    ComponentExecutable,
    End,
    Start,
    Workflow,
    Input,
    Output,
    create_workflow_session,
 )

---
## 1️⃣ 节点定义

In [3]:
class CheckBalanceNode(ComponentExecutable, ComponentComposable):
    """Mock：检查账户余额，如果 < 0.01 USD 则强制让用户确认"""
    async def invoke(self, inputs: Input, session: Session, context: ModelContext) -> Output:
        balance = random.uniform(0.005, 0.02)   # 随机模拟余额
        print(f"[CheckBalance] 当前余额: ${balance:.4f}")
        if balance < 0.01:
            confirm = await session.interact(
                "调用天气 API 需要 $0.01，余额可能不足，是否继续？（Yes/No）"
            )
            if confirm.lower() != "yes":
                raise Exception("用户拒绝付费，流程终止")
        return {"balance_ok": True}

In [4]:
class AskCityNode(ComponentExecutable, ComponentComposable):
    async def invoke(self, inputs: Input, session: Session, context: ModelContext) -> Output:
        city = await session.interact("请输入要查询的城市（如 Beijing）：")
        return {"city": city.strip()}

In [5]:
class AskUnitNode(ComponentExecutable, ComponentComposable):
    async def invoke(self, inputs: Input, session: Session, context: ModelContext) -> Output:
        unit = await session.interact("温度单位选 Celsius 还是 Fahrenheit？（C/F）")
        unit = unit.strip().upper()
        if unit not in {"C", "F"}:
            unit = "C"
        return {"unit": unit}

In [6]:
class QueryWeatherNode(ComponentExecutable, ComponentComposable):
    """Mock：50% 概率抛网络异常，第二次重试一定成功"""
    def __init__(self):
        super().__init__()
        self.tried = False

    async def invoke(self, inputs: Input, session: Session, context: ModelContext) -> Output:
        city = inputs.get("city")
        unit = inputs.get("unit")
        print(f"[QueryWeather] 开始查询 {city} 单位={unit}")

        # 第一次 50% 概率失败
        if not self.tried and random.random() < 0.5:
            self.tried = True
            raise Exception("网络超时：连接 openweathermap.org 失败")

        # Mock 返回
        temp = random.randint(-10, 35)
        if unit == "F":
            temp = temp * 9 // 5 + 32
        summary = "sunny" if temp > 20 else "cloudy"
        return {"weather": f"{city} 明天 {temp}°{unit} {summary}"}

---
## 2️⃣ 构图

In [7]:
def build_weather_agent():
    flow = Workflow()
    flow.set_start_comp("start", Start(), inputs_schema={"trigger": "${trigger}"})

    flow.add_workflow_comp("check", CheckBalanceNode())

    flow.add_workflow_comp("ask_city", AskCityNode(),
                           outputs_schema={"city": "${city}"})
    flow.add_workflow_comp("ask_unit", AskUnitNode(),
                           outputs_schema={"unit": "${unit}"})

    flow.add_workflow_comp("weather", QueryWeatherNode(),
                           inputs_schema={"city": "${ask_city.city}", "unit": "${ask_unit.unit}"},
                           outputs_schema={"report": "${weather}"})

    flow.set_end_comp("end", End(),
                      inputs_schema={"report": "${weather.report}"})

    flow.add_connection("start", "check")
    flow.add_connection("check", "ask_city")
    flow.add_connection("check", "ask_unit")
    flow.add_connection("ask_city", "weather")
    flow.add_connection("ask_unit", "weather")
    flow.add_connection("weather", "end")
    return flow

---
## 3️⃣ 运行：第一次可能触发余额确认 + 网络异常

In [8]:
agent = build_weather_agent()
session_id = uuid.uuid4().hex
print("🌀 会话ID =", session_id)

🌀 会话ID = c07952f0097b4bf2b128913cca847b93


In [9]:
# 第一次调用 → 可能被余额检查中断，也可能被网络异常中断
try:
    out = await agent.invoke(
        {"trigger": "明天我要出差，帮我查天气"},
        create_workflow_session(session_id=session_id)
    )
    print("☀️ 最终结果：", out.result.get("output").get("report"))
except Exception as e:
    print("💥 第一次执行被中断：", e)

2026-02-25 22:47:35 | workflow | workflow.py | 268 | invoke | default_trace_id | INFO | {"event_id": "79c38389-307e-47e0-946d-0b20f1367ab9", "event_type": "workflow_execute_start", "log_level": "INFO", "timestamp": "2026-02-25T14:47:35.464582+00:00", "module_type": "workflow", "module_id": "workflow", "module_name": "workflow", "status": "success", "message": "Begin to run workflow invoke", "metadata": {}, "workflow_id": "e14477c1fab74d2aad69d8f6303293a6", "workflow_name": "", "inputs": {"trigger": "明天我要出差，帮我查天气"}}
2026-02-25 22:47:35 | graph | vertex.py | 63 | init | default_trace_id | INFO | {"event_id": "64f1d357-9726-4641-aa0e-7b70926a7c72", "event_type": "graph_vertex_init", "log_level": "INFO", "timestamp": "2026-02-25T14:47:35.470654+00:00", "module_type": "system", "module_id": "graph", "module_name": "graph", "status": "success", "message": "Initialized node [start], abilities is ['invoke']", "metadata": {}, "graph_id": "e14477c1fab74d2aad69d8f6303293a6", "node_id": "start"}
2

根据中断类型，下面分别处理：

In [10]:
# 如果是交互中断，out.result 里会携带问题
if 'out' in locals() and hasattr(out, 'result') and out.result and isinstance(out.result, list):
    for q in out.result:
        print("❓", q.payload.value)

❓ 温度单位选 Celsius 还是 Fahrenheit？（C/F）
❓ 请输入要查询的城市（如 Beijing）：


In [11]:
# 构造用户回答（余额确认 + 城市 + 单位）
user_input = InteractiveInput()

# 1. 余额确认（若曾被问）
for q in out.result:
    if "余额" in q.payload.value:
        user_input.update(q.payload.id, "Yes")   # 同意付费

# 2. 城市 & 单位（若曾被问）
for q in out.result:
    if "城市" in q.payload.value:
        user_input.update(q.payload.id, "Shanghai")
    elif "温度单位" in q.payload.value:
        user_input.update(q.payload.id, "C")

In [12]:
# 恢复运行（若第一次已 success 则本格直接拿到结果）
final = await agent.invoke(user_input, create_workflow_session(session_id=session_id))
print("☀️ 最终结果：", final.result.get("output").get("report"))

2026-02-25 22:47:44 | workflow | workflow.py | 268 | invoke | default_trace_id | INFO | {"event_id": "05fb45d9-73a7-43c5-8d03-123268ef7881", "event_type": "workflow_execute_start", "log_level": "INFO", "timestamp": "2026-02-25T14:47:44.797792+00:00", "module_type": "workflow", "module_id": "workflow", "module_name": "workflow", "status": "success", "message": "Begin to run workflow invoke", "metadata": {}, "workflow_id": "e14477c1fab74d2aad69d8f6303293a6", "workflow_name": "", "inputs": "user_inputs={'ask_unit': 'C', 'ask_city': 'Shanghai'} raw_inputs=None"}
2026-02-25 22:47:44 | session | inmemory.py | 45 | pre_workflow_execute | default_trace_id | INFO | {"event_id": "456e6afb-b5e3-4db3-92e4-641500e1e6de", "event_type": "checkpoint_restore", "log_level": "INFO", "timestamp": "2026-02-25T14:47:44.800305+00:00", "module_type": "session", "module_id": "session", "module_name": "session", "session_id": "c07952f0097b4bf2b128913cca847b93", "status": "success", "message": "Begin to restore 

---
## 4️⃣ 再跑一次：余额检查不会再执行（已成功节点跳过）

In [13]:
# 全新会话，观察「余额检查」会再跑一次；但旧会话重试不会跑
new_session = uuid.uuid4().hex
print("🌀 新会话ID =", new_session)

try:
    await agent.invoke({"trigger": "查天气"}, create_workflow_session(session_id=new_session))
except Exception as e:
    print("💥 新会话第一次中断：", e)

🌀 新会话ID = 180857d2f01742e78d38e9dc0aa8bcd1
2026-02-25 22:47:48 | workflow | workflow.py | 268 | invoke | default_trace_id | INFO | {"event_id": "ab29c337-cef0-4c21-a485-b6b4f5ff8ff6", "event_type": "workflow_execute_start", "log_level": "INFO", "timestamp": "2026-02-25T14:47:48.006673+00:00", "module_type": "workflow", "module_id": "workflow", "module_name": "workflow", "status": "success", "message": "Begin to run workflow invoke", "metadata": {}, "workflow_id": "e14477c1fab74d2aad69d8f6303293a6", "workflow_name": "", "inputs": {"trigger": "查天气"}}
2026-02-25 22:47:48 | session | inmemory.py | 40 | pre_workflow_execute | default_trace_id | INFO | {"event_id": "9184c345-d767-4f3d-a44d-5b9b33eceeaf", "event_type": "checkpointer_store_add", "log_level": "INFO", "timestamp": "2026-02-25T14:47:48.007831+00:00", "module_type": "session", "module_id": "session", "module_name": "session", "session_id": "180857d2f01742e78d38e9dc0aa8bcd1", "status": "success", "message": "Create a new workflow c

---
## ✅ 总结

| 能力 | 用户视角 | 开发者视角 |
|---|---|---|
| 交互中断 | 弹窗确认 / 输入城市 | 同一 `session_id` + `InteractiveInput` 回复即可续跑 |
| 异常中断 | 网络失败提示重试 | 抛异常后重试，只重跑失败节点，已成功的「余额检查」不再扣费 |
| 多节点并发 | 一次问多个问题 | 为不同 `node_id` 分别 `update()` 回答 |

把这段代码直接搬进项目，你就拥有了一个 **会省钱、会容错、会等人** 的智能天气 Agent！